# AtliQ Hardware — Retail Sales ETL Pipeline
### Built with Databricks & PySpark
**Author:** Sree Lalitha Prudhvi | Data Analyst

This notebook builds an end-to-end ETL pipeline:
- **Extract:** Load raw sales data
- **Transform:** Clean, join and calculate using PySpark
- **Load:** Save clean tables ready for analysis

## Step 1 — Extract: Load Raw Sales Data

In [0]:
# Extract: Create raw customer data
customer_data = [
    (70002017, "Atliq Exclusive", "Brick & Mortar", "India", "APAC"),
    (70002018, "Atliq e Store", "E-Commerce", "India", "APAC"),
    (70002019, "Flipkart", "E-Commerce", "India", "APAC"),
    (70002039, "Atliq Exclusive", "Brick & Mortar", "USA", "NA"),
    (70002040, "Atliq e Store", "E-Commerce", "USA", "NA"),
    (70002048, "Atliq Exclusive", "Brick & Mortar", "UK", "EU"),
    (70002051, "Amazon", "E-Commerce", "France", "EU"),
    (70002027, "Reliance Digital", "Brick & Mortar", "India", "APAC"),
    (70002064, "Atliq e Store", "E-Commerce", "Mexico", "LATAM"),
    (70002071, "Amazon", "E-Commerce", "Italy", "EU")
]

customer_columns = ["customer_code", "customer", "platform", "market", "region"]

df_customer = spark.createDataFrame(customer_data, customer_columns)
df_customer.show()

+-------------+----------------+--------------+------+------+
|customer_code|        customer|      platform|market|region|
+-------------+----------------+--------------+------+------+
|     70002017| Atliq Exclusive|Brick & Mortar| India|  APAC|
|     70002018|   Atliq e Store|    E-Commerce| India|  APAC|
|     70002019|        Flipkart|    E-Commerce| India|  APAC|
|     70002039| Atliq Exclusive|Brick & Mortar|   USA|    NA|
|     70002040|   Atliq e Store|    E-Commerce|   USA|    NA|
|     70002048| Atliq Exclusive|Brick & Mortar|    UK|    EU|
|     70002051|          Amazon|    E-Commerce|France|    EU|
|     70002027|Reliance Digital|Brick & Mortar| India|  APAC|
|     70002064|   Atliq e Store|    E-Commerce|Mexico| LATAM|
|     70002071|          Amazon|    E-Commerce| Italy|    EU|
+-------------+----------------+--------------+------+------+



In [0]:
# Extract: Create raw sales data
sales_data = [
    ("2020-01-01", 70002017, "P001", 100, 500.0),
    ("2020-01-01", 70002018, "P002", 150, 300.0),
    ("2020-02-01", 70002019, "P001", 200, 500.0),
    ("2020-02-01", 70002039, "P003", 120, 750.0),
    ("2020-03-01", 70002040, "P002", 180, 300.0),
    ("2020-03-01", 70002048, "P001", 90, 500.0),
    ("2020-04-01", 70002051, "P003", 210, 750.0),
    ("2020-04-01", 70002027, "P002", 160, 300.0),
    ("2020-05-01", 70002064, "P001", 140, 500.0),
    ("2020-05-01", 70002071, "P003", 175, 750.0),
    ("2020-06-01", 70002017, "P002", 220, 300.0),
    ("2020-06-01", 70002018, "P001", 130, 500.0)
]

sales_columns = ["date", "customer_code", "product_code", "sold_quantity", "unit_price"]

df_sales = spark.createDataFrame(sales_data, sales_columns)
df_sales.show()

+----------+-------------+------------+-------------+----------+
|      date|customer_code|product_code|sold_quantity|unit_price|
+----------+-------------+------------+-------------+----------+
|2020-01-01|     70002017|        P001|          100|     500.0|
|2020-01-01|     70002018|        P002|          150|     300.0|
|2020-02-01|     70002019|        P001|          200|     500.0|
|2020-02-01|     70002039|        P003|          120|     750.0|
|2020-03-01|     70002040|        P002|          180|     300.0|
|2020-03-01|     70002048|        P001|           90|     500.0|
|2020-04-01|     70002051|        P003|          210|     750.0|
|2020-04-01|     70002027|        P002|          160|     300.0|
|2020-05-01|     70002064|        P001|          140|     500.0|
|2020-05-01|     70002071|        P003|          175|     750.0|
|2020-06-01|     70002017|        P002|          220|     300.0|
|2020-06-01|     70002018|        P001|          130|     500.0|
+----------+-------------

## Step 2 — Transform: Clean, Join & Calculate

In [0]:
# Transform 1: Calculate revenue (quantity x price)
from pyspark.sql.functions import col, round

df_sales_transformed = df_sales.withColumn(
    "revenue",
    round(col("sold_quantity") * col("unit_price"), 2)
)

df_sales_transformed.show()

+----------+-------------+------------+-------------+----------+--------+
|      date|customer_code|product_code|sold_quantity|unit_price| revenue|
+----------+-------------+------------+-------------+----------+--------+
|2020-01-01|     70002017|        P001|          100|     500.0| 50000.0|
|2020-01-01|     70002018|        P002|          150|     300.0| 45000.0|
|2020-02-01|     70002019|        P001|          200|     500.0|100000.0|
|2020-02-01|     70002039|        P003|          120|     750.0| 90000.0|
|2020-03-01|     70002040|        P002|          180|     300.0| 54000.0|
|2020-03-01|     70002048|        P001|           90|     500.0| 45000.0|
|2020-04-01|     70002051|        P003|          210|     750.0|157500.0|
|2020-04-01|     70002027|        P002|          160|     300.0| 48000.0|
|2020-05-01|     70002064|        P001|          140|     500.0| 70000.0|
|2020-05-01|     70002071|        P003|          175|     750.0|131250.0|
|2020-06-01|     70002017|        P002

In [0]:
# Transform 2: Join sales with customer data
df_joined = df_sales_transformed.join(
    df_customer,
    on="customer_code",
    how="left"
)

df_joined.show()

+-------------+----------+------------+-------------+----------+--------+----------------+--------------+------+------+
|customer_code|      date|product_code|sold_quantity|unit_price| revenue|        customer|      platform|market|region|
+-------------+----------+------------+-------------+----------+--------+----------------+--------------+------+------+
|     70002017|2020-01-01|        P001|          100|     500.0| 50000.0| Atliq Exclusive|Brick & Mortar| India|  APAC|
|     70002018|2020-01-01|        P002|          150|     300.0| 45000.0|   Atliq e Store|    E-Commerce| India|  APAC|
|     70002019|2020-02-01|        P001|          200|     500.0|100000.0|        Flipkart|    E-Commerce| India|  APAC|
|     70002039|2020-02-01|        P003|          120|     750.0| 90000.0| Atliq Exclusive|Brick & Mortar|   USA|    NA|
|     70002040|2020-03-01|        P002|          180|     300.0| 54000.0|   Atliq e Store|    E-Commerce|   USA|    NA|
|     70002048|2020-03-01|        P001| 

## Step 3 — Aggregations: Business Insights

In [0]:
# Aggregation 1: Total revenue by market
from pyspark.sql.functions import sum as _sum, round

df_market_revenue = df_joined.groupBy("market") \
    .agg(round(_sum("revenue"), 2).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

df_market_revenue.show()

+------+-------------+
|market|total_revenue|
+------+-------------+
| India|     374000.0|
|France|     157500.0|
|   USA|     144000.0|
| Italy|     131250.0|
|Mexico|      70000.0|
|    UK|      45000.0|
+------+-------------+



In [0]:
# Aggregation 2: Total revenue by region
df_region_revenue = df_joined.groupBy("region") \
    .agg(round(_sum("revenue"), 2).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

df_region_revenue.show()

+------+-------------+
|region|total_revenue|
+------+-------------+
|  APAC|     374000.0|
|    EU|     333750.0|
|    NA|     144000.0|
| LATAM|      70000.0|
+------+-------------+



In [0]:
# Aggregation 3: Total revenue by platform
df_platform_revenue = df_joined.groupBy("platform") \
    .agg(round(_sum("revenue"), 2).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

df_platform_revenue.show()

+--------------+-------------+
|      platform|total_revenue|
+--------------+-------------+
|    E-Commerce|     622750.0|
|Brick & Mortar|     299000.0|
+--------------+-------------+



## Step 4 — Load: Save Clean Tables

In [0]:
# Load: Save transformed data as tables
df_joined.write.mode("overwrite").saveAsTable("atliq_sales_clean")
df_market_revenue.write.mode("overwrite").saveAsTable("atliq_market_revenue")

print("Tables saved successfully!")

Tables saved successfully!


In [0]:
%sql
SELECT * FROM atliq_sales_clean LIMIT 10

customer_code,date,product_code,sold_quantity,unit_price,revenue,customer,platform,market,region
70002017,2020-06-01,P002,220,300.0,66000.0,Atliq Exclusive,Brick & Mortar,India,APAC
70002018,2020-06-01,P001,130,500.0,65000.0,Atliq e Store,E-Commerce,India,APAC
70002027,2020-04-01,P002,160,300.0,48000.0,Reliance Digital,Brick & Mortar,India,APAC
70002064,2020-05-01,P001,140,500.0,70000.0,Atliq e Store,E-Commerce,Mexico,LATAM
70002040,2020-03-01,P002,180,300.0,54000.0,Atliq e Store,E-Commerce,USA,NA
70002048,2020-03-01,P001,90,500.0,45000.0,Atliq Exclusive,Brick & Mortar,UK,EU
70002018,2020-01-01,P002,150,300.0,45000.0,Atliq e Store,E-Commerce,India,APAC
70002019,2020-02-01,P001,200,500.0,100000.0,Flipkart,E-Commerce,India,APAC
70002039,2020-02-01,P003,120,750.0,90000.0,Atliq Exclusive,Brick & Mortar,USA,NA
70002051,2020-04-01,P003,210,750.0,157500.0,Amazon,E-Commerce,France,EU


## Step 5 — Visualisation: Revenue Insights

In [0]:
%sql
SELECT market, SUM(revenue) AS total_revenue
FROM atliq_sales_clean
GROUP BY market
ORDER BY total_revenue DESC

market,total_revenue
India,374000.0
France,157500.0
USA,144000.0
Italy,131250.0
Mexico,70000.0
UK,45000.0


Databricks visualization. Run in Databricks to view.

## Project Summary

This ETL pipeline demonstrates an end-to-end data workflow built in Databricks using PySpark:

**Extract:** Loaded raw customer and sales data into Spark DataFrames.

**Transform:**
- Calculated revenue (quantity × unit price)
- Joined sales data with customer dimension data
- Aggregated revenue by market, region and platform

**Load:** Saved clean, transformed data as queryable tables.

**Visualisation:** Created revenue insights using SQL and built-in charts.

### Key Skills Demonstrated
- PySpark (DataFrames, joins, aggregations)
- ETL pipeline design
- Mixing Python and SQL in Databricks
- Data transformation and cleaning

### Tools Used
Databricks | PySpark | Spark SQL | Python